# Chapter 5 — Bayesian Prediction: Forecasts and Uncertainty

This notebook contains **two complementary synthetic figures**:

1. **Option A (new):** a visually interpretable posterior predictive forecast, showing historical observations, the forecast origin, the posterior mean, pointwise 50% and 95% intervals for *latent position*, sample futures, and held-out observations. The forecast only uses data at or before 2 s.
2. **Variance decomposition (original):** the chapter's existing numerical variance-contribution plot, computed using the original prior predictive model. It remains available for comparison.

The Option A illustration extends the chapter's constant-acceleration example. Let \(x(t)=Vt+\tfrac12 At^2\), with independent priors \(V\sim\mathcal N(1.2,0.04)\) m/s and \(A\sim\mathcal N(0,0.09)\) m/s\(^2\), where the second parameter denotes variance. Eight synthetic noisy positions through 2 s update the jointly Gaussian distribution of \((V,A)\). Later observations are held out for visualization, not assimilation. All data are **synthetic**, generated with a fixed random seed; the figure does not claim real-world validation.


In [ ]:
from pathlib import Path
import json, math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from IPython.display import display, Image

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
outdir = root / 'figures'
outdir.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(20261114)

# Model: x(t) = V*t + 0.5*A*t**2; independent Gaussian priors.
prior_mean = np.array([1.2, 0.0])  # m/s, m/s**2
prior_cov = np.diag([0.04, 0.09])
sigma_obs = 0.12  # position observation SD [m]
def basis(t):
    t = np.asarray(t)
    return np.column_stack((t, 0.5*t**2))

true_params = rng.multivariate_normal(prior_mean, prior_cov)
t_obs = np.linspace(0.25, 2.0, 8)
t_held = np.array([2.35, 2.75, 3.2, 3.7, 4.25, 4.8])
y_obs = basis(t_obs) @ true_params + rng.normal(0, sigma_obs, t_obs.size)
y_held = basis(t_held) @ true_params + rng.normal(0, sigma_obs, t_held.size)
assert t_obs.max() == 2.0 and np.all(t_held > 2.0)

# Exact linear-Gaussian posterior; no future observations enter this update.
H = basis(t_obs)
post_cov = np.linalg.inv(np.linalg.inv(prior_cov) + H.T @ H / sigma_obs**2)
post_mean = post_cov @ (np.linalg.solve(prior_cov, prior_mean)
                        + H.T @ y_obs / sigma_obs**2)
t_forecast = np.linspace(2.0, 5.0, 240)
t_full = np.linspace(0.0, 5.0, 450)
B = basis(t_forecast)
forecast_mean = B @ post_mean
forecast_sd = np.sqrt(np.einsum('ij,jk,ik->i', B, post_cov, B))
sample_params = rng.multivariate_normal(post_mean, post_cov, 7)
assert np.all(np.linalg.eigvalsh(post_cov) > 0)
assert np.all(forecast_sd >= 0)

# Berkeley blue/gold, book-sized PDF with embedded vector text.
blue, gold, cyan = '#003262', '#FDB515', '#1C86A2'
with plt.rc_context({'font.family':'serif','font.size':10.8,'mathtext.fontset':'cm',
                     'axes.edgecolor':'#ACB4BD', 'axes.labelcolor':'#272D37',
                     'xtick.color':'#4F5864','ytick.color':'#4F5864',
                     'pdf.fonttype':42,'ps.fonttype':42}):
    fig, ax = plt.subplots(figsize=(6.5, 4.6))
    fig.subplots_adjust(left=.105, right=.975, bottom=.32, top=.95)
    ax.axvspan(0, 2, color='#F0F4F8', zorder=-2)
    ax.fill_between(t_forecast, forecast_mean-1.96*forecast_sd,
                    forecast_mean+1.96*forecast_sd, color=blue, alpha=.21,
                    label='95% credible band',zorder=1)
    ax.fill_between(t_forecast, forecast_mean-.67448975*forecast_sd,
                    forecast_mean+.67448975*forecast_sd, color=cyan, alpha=.40,
                    label='50% credible band',zorder=2)
    for j, params in enumerate(sample_params):
        ax.plot(t_forecast,basis(t_forecast)@params,color=cyan,lw=.78,alpha=.46,
                label='Posterior sample trajectories' if j==0 else None,zorder=3)
    ax.plot(t_forecast,forecast_mean,color=blue,lw=2.5,
            label='Posterior predictive mean',zorder=6)
    ax.plot(t_full,basis(t_full)@true_params,color='#666F7D',
            linestyle=(0,(4,3)),lw=1.65,label='Synthetic latent truth',zorder=5)
    ax.scatter(t_obs,y_obs,s=34,color=blue,edgecolor='white',lw=.6,
               label='Assimilated observations',zorder=9)
    ax.scatter(t_held,y_held,s=51,marker='D',color=gold,edgecolor='#6B5200',
               lw=.7,label='Held-out observations',zorder=8)
    ax.axvline(2,color='#C07A00',lw=1.65,linestyle=(0,(5,3)),zorder=7)
    ax.text(2.04,.975,'Forecast origin',transform=ax.get_xaxis_transform(),
            ha='left',va='top',color='#926000',fontsize=10)
    ax.text(.06,.07,'Observed',transform=ax.transAxes,ha='left',
            color='#526272',fontsize=10)
    ax.text(.95,.07,'Forecast',transform=ax.transAxes,ha='right',
            color='#526272',fontsize=9)
    ax.set(xlim=(0,5), xlabel='Time [s]', ylabel='Position [m]')
    ax.set_ylim(bottom=min(-.9,float((forecast_mean-1.96*forecast_sd).min())-.15))
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y',color='#DDE3EA',alpha=.55,lw=.65)
    ax.set_axisbelow(True)
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    handles,labels = ax.get_legend_handles_labels()
    legend_order=['Assimilated observations','Held-out observations',
                  'Posterior predictive mean','95% credible band',
                  '50% credible band','Posterior sample trajectories',
                  'Synthetic latent truth']
    legend_names=['Observed data','Held-out data','Predictive mean',
                  '95% credible band','50% credible band','Sample futures',
                  'Latent truth']
    fig.legend([handles[labels.index(s)] for s in legend_order],legend_names,
               ncol=3,loc='lower center',bbox_to_anchor=(.52,.005),
               frameon=False,fontsize=10.2,columnspacing=1.0,
               handlelength=1.9,handletextpad=.45)
    fig.savefig(outdir/'bayesian-prediction-forecast.png',dpi=250)
    fig.savefig(outdir/'bayesian-prediction-forecast.pdf',
                metadata={'Creator':'STAIRlab Chapter 5 synthetic forecast'})
    plt.close(fig)

print('Posterior mean [V, A]:', post_mean)
print('Posterior covariance:', post_cov)
print('Forecast plot:', outdir/'bayesian-prediction-forecast.pdf')
display(Image(filename=str(outdir/'bayesian-prediction-forecast.png')))


## Interpretation and limitations

The shaded regions show **pointwise central posterior intervals for latent position**, not simultaneous trajectory-wide bands or future noisy-measurement intervals. The dashed gray curve is the synthetic latent truth; gold diamonds are held-out noisy observations.

The model assumes a fixed initial position, time-constant but uncertain velocity and acceleration, independent Gaussian priors, and independent Gaussian measurement noise. Under dependence between V and A, a covariance contribution appears. Held-out observations are **not used** to fit the posterior. Their visual agreement does not establish coverage or model adequacy.


## Original Chapter 5 variance-decomposition illustration

This existing figure decomposes the **prior predictive** position variance into velocity and acceleration contributions: 0.04 t² + 0.0225 t⁴. It remains reproducible and is independent of the observation-conditioned forecast above.


In [ ]:
from pathlib import Path
import sys, json, math
from IPython.display import Image, display

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
from robotics_book.chapter_examples import run_example

# Independently verify the chapter's variance decomposition at t = 3 s.
t = 3.0
var_velocity = (t**2) * 0.04
var_acceleration = (0.5 * t**2)**2 * 0.09
var_total = var_velocity + var_acceleration
assert math.isclose(var_velocity, 0.36)
assert math.isclose(var_acceleration, 1.8225)
assert math.isclose(var_total, 2.1825)

result = run_example('bayesian-prediction', root=root)
assert math.isclose(result['values']['mean_at_3'], 3.6)
assert math.isclose(result['values']['variance_at_3'], var_total)
print(json.dumps(result['values'], indent=2))
display(Image(filename=str(root / 'figures' / 'bayesian-prediction.png')))


At t = 3 s, mean position is 3.6 m and total prior predictive variance is 2.1825 m². This figure and the new Option A plot answer different questions: *prior uncertainty growth* versus *prediction after assimilating data*. The original manuscript import should remain unchanged until the revised figure has been reviewed.
